# 4. Optimización simbólica de las ecuaciones geodésicas de Kerr

Este cuaderno construye el lado derecho de la ecuación geodésica directamente a partir de los símbolos de Christoffel explícitos de la instantánea legada de RelatiPy y aplica eliminación de subexpresiones comunes (CSE) **después** de la contracción. También deriva y optimiza la cuadrivelocidad temporal a partir de las tres velocidades coordenadas. El C generado conserva ambas reducciones algebraicas, pero sustituye los nombres opacos de CSE por la notación semántica usada en el backend (`r2`, `spin2`, `cos_th`, `sigma`, etc.).

> **Alcance.** La optimización es algebraica y heurística; minimiza el conteo de operaciones entre varias estrategias ensayadas, pero no demuestra optimalidad global. Las expresiones de Christoffel se reproducen literalmente desde el método legado solicitado y conservan sus discrepancias históricas conocidas con la métrica corregida. Este cuaderno no modifica el backend ni constituye por sí solo una implementación de producción.

## 4.1. Sistema de primer orden

Para coordenadas de Boyer--Lindquist $x^\mu=(t,r,\theta,\phi)$ y cuadrivelocidad $u^\mu=dx^\mu/d\lambda$, la ecuación geodésica afín es

$$\frac{d x^\mu}{d\lambda}=u^\mu,\qquad\frac{d u^\mu}{d\lambda}=-\Gamma^\mu{}_{\alpha\beta}u^\alpha u^\beta.$$

La simetría $\Gamma^\mu{}_{\alpha\beta}=\Gamma^\mu{}_{\beta\alpha}$ permite contraer solamente el triángulo superior: términos diagonales una vez y términos fuera de la diagonal con factor dos. Para geodésicas temporales, $\lambda$ puede elegirse como tiempo propio.

In [34]:
from __future__ import annotations

import ast
import ctypes
import importlib.util
import shutil
import subprocess
import tempfile
import time
from itertools import product
from pathlib import Path

import numpy as np
import sympy as sp


def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'relatipy-legacy-local-2026-09-01').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate the thesis project root')


ROOT = find_project_root()
LEGACY_ROOT = ROOT / 'relatipy-legacy-local-2026-09-01'
LEGACY_KERR = LEGACY_ROOT / 'src/relatipy/numeric/metrics/kerr_metric.py'
LEGACY_OPTIMIZER = LEGACY_ROOT / 'notebooks/utils/symbolic_to_numeric.py'
assert LEGACY_KERR.is_file() and LEGACY_OPTIMIZER.is_file()
ROOT

PosixPath('/home/lldddv/Documents/UdeA/thesis')

## 4.2. Utilidad legada de optimización

Se carga el script legado en lugar de copiarlo. Su estrategia identifica expresiones repetidas, genera nombres válidos y conserva solamente los componentes independientes del tensor. Aquí reutilizamos su normalización de nombres y extendemos la idea: primero contraemos $\Gamma$ con $u^\alpha u^\beta$, luego ejecutamos CSE sobre las cuatro aceleraciones simultáneamente y finalmente renombramos cada auxiliar con una descripción algebraica legible.

In [35]:
def load_module(path: Path, name: str):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise ImportError(f'Cannot load {path}')
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


legacy_codegen = load_module(LEGACY_OPTIMIZER, 'legacy_symbolic_to_numeric')
assert legacy_codegen.make_valid_varname(r'\dot{x}^1') == 'dx1_dt'
legacy_codegen.make_valid_varname('sin(theta)**2')

'sin_IthetaI__pow2'

## 4.3. Extracción simbólica de la fuente canónica legada

La función siguiente analiza el AST de `Kerr._get_christoffel_symbols` y ejecuta únicamente su rama escalar, desde la asignación de coordenadas hasta antes del `return`. `numpy.zeros` se reemplaza por un arreglo mutable de SymPy y las funciones trigonométricas por sus equivalentes simbólicos. La entrada es un archivo local confiable y congelado; no debe usarse este mecanismo con código arbitrario.

In [36]:
class SymbolicNumpy:
    sin = staticmethod(sp.sin)
    cos = staticmethod(sp.cos)

    @staticmethod
    def zeros(shape):
        return sp.MutableDenseNDimArray.zeros(*shape)

    @staticmethod
    def diag(values):
        return sp.MutableDenseMatrix.diag(*values)


class SymbolicKerrParameters:
    def __init__(self, spin, schwarzschild_radius):
        self.a = spin
        self.R_s = schwarzschild_radius


def extract_legacy_christoffel(path: Path, coordinates, spin, rs):
    tree = ast.parse(path.read_text(encoding='utf-8'), filename=str(path))
    kerr_class = next(
        node for node in tree.body
        if isinstance(node, ast.ClassDef) and node.name == 'Kerr'
    )
    method = next(
        node for node in kerr_class.body
        if isinstance(node, ast.FunctionDef)
        and node.name == '_get_christoffel_symbols'
    )
    scalar_start = next(
        index for index, statement in enumerate(method.body)
        if isinstance(statement, ast.Assign)
        and any(isinstance(target, ast.Tuple) for target in statement.targets)
    )
    scalar_body = []
    for statement in method.body[scalar_start:]:
        if isinstance(statement, ast.Return):
            break
        scalar_body.append(statement)

    namespace = {
        'self': SymbolicKerrParameters(spin, rs),
        'xs': coordinates,
        'numpy': SymbolicNumpy,
        'sin': sp.sin,
        'cos': sp.cos,
        'tan': sp.tan,
    }
    module = ast.fix_missing_locations(ast.Module(body=scalar_body, type_ignores=[]))
    exec(compile(module, str(path), 'exec'), namespace)
    return sp.ImmutableDenseNDimArray(namespace['Gamma'])


def extract_legacy_metric(path: Path, coordinates, spin, rs):
    tree = ast.parse(path.read_text(encoding='utf-8'), filename=str(path))
    kerr_class = next(
        node for node in tree.body
        if isinstance(node, ast.ClassDef) and node.name == 'Kerr'
    )
    method = next(
        node for node in kerr_class.body
        if isinstance(node, ast.FunctionDef)
        and node.name == '_metric_dimensionless'
    )
    scalar_branch = next(
        statement for statement in method.body
        if isinstance(statement, ast.If)
    )
    scalar_body = []
    for statement in scalar_branch.body:
        if isinstance(statement, ast.Return):
            break
        scalar_body.append(statement)

    namespace = {
        'self': SymbolicKerrParameters(spin, rs),
        'xs': coordinates,
        'numpy': SymbolicNumpy,
    }
    module = ast.fix_missing_locations(ast.Module(body=scalar_body, type_ignores=[]))
    exec(compile(module, str(path), 'exec'), namespace)
    return sp.ImmutableDenseMatrix(namespace['metric'])


In [37]:
t, r, th, ph = sp.symbols('t r th ph', real=True)
rs, spin = sp.symbols('rs spin', real=True)
u = sp.symbols('u0:4', real=True)
coordinates = (t, r, th, ph)

Gamma = extract_legacy_christoffel(LEGACY_KERR, coordinates, spin, rs)
nonzero_components = [
    (mu, alpha, beta)
    for mu, alpha, beta in product(range(4), repeat=3)
    if Gamma[mu, alpha, beta] != 0
]
assert Gamma.shape == (4, 4, 4)
assert len(nonzero_components) == 32
assert all(
    sp.simplify(Gamma[mu, alpha, beta] - Gamma[mu, beta, alpha]) == 0
    for mu, alpha, beta in product(range(4), repeat=3)
)
{'shape': Gamma.shape, 'nonzero_components': len(nonzero_components)}

{'shape': (4, 4, 4), 'nonzero_components': 32}

## 4.4. Contracción antes de optimizar

La contracción simétrica evita evaluar dos veces cada par de índices inferiores. Se construyen también las expresiones completas de 16 términos por componente para comprobar simbólicamente que ambas formas coinciden.

In [38]:
direct_acceleration = tuple(
    -sum(
        Gamma[mu, alpha, beta] * u[alpha] * u[beta]
        for alpha, beta in product(range(4), repeat=2)
    )
    for mu in range(4)
)

symmetric_acceleration = tuple(
    -sum(Gamma[mu, alpha, alpha] * u[alpha] ** 2 for alpha in range(4))
    - 2 * sum(
        Gamma[mu, alpha, beta] * u[alpha] * u[beta]
        for alpha in range(4)
        for beta in range(alpha + 1, 4)
    )
    for mu in range(4)
)

contraction_residuals = tuple(
    sp.simplify(direct - symmetric)
    for direct, symmetric in zip(direct_acceleration, symmetric_acceleration)
)
assert contraction_residuals == (0, 0, 0, 0)
first_order_rhs = (*u, *symmetric_acceleration)
[sp.count_ops(expression) for expression in symmetric_acceleration]

[113, 290, 182, 156]

## 4.5. Selección reproducible y nombres semánticos

Se comparan CSE directa, CSE con optimizaciones básicas y las mismas dos variantes después de `factor_terms`. El criterio es el número total de operaciones SymPy de los auxiliares más las cuatro expresiones finales. Los desempates favorecen menos auxiliares y luego el nombre de la estrategia. La estrategia ganadora se renombra de forma determinista con la convención del evaluador de Christoffel: el número unido al símbolo indica potencia (`r2`, `cos2_th`) y el guion bajo distingue un ángulo múltiple (`sin_2th`).

In [39]:
def operation_cost(replacements, expressions) -> int:
    return int(
        sum(sp.count_ops(value) for _, value in replacements)
        + sum(sp.count_ops(expression) for expression in expressions)
    )


candidate_inputs = {
    'cse': symmetric_acceleration,
    'factor_terms+cse': tuple(sp.factor_terms(expr) for expr in symmetric_acceleration),
}
strategies = []
for input_name, expressions in candidate_inputs.items():
    for optimization_name, optimizations in (('raw', None), ('basic', 'basic')):
        started = time.perf_counter()
        replacements, reduced = sp.cse(
            expressions,
            symbols=sp.numbered_symbols('cse_aux'),
            optimizations=optimizations,
            order='canonical',
        )
        strategies.append({
            'name': f'{input_name}:{optimization_name}',
            'replacements': tuple(replacements),
            'reduced': tuple(reduced),
            'temporaries': len(replacements),
            'operations': operation_cost(replacements, reduced),
            'elapsed_s': time.perf_counter() - started,
        })

best_cse = min(
    strategies,
    key=lambda item: (item['operations'], item['temporaries'], item['name']),
)
baseline_operations = int(sum(sp.count_ops(expr) for expr in symmetric_acceleration))
semantic_names = (
    'cos_th', 'r2', 'spin2', 'cos2_th', 'spin2_cos2_th', 'sigma',
    'inv_sigma2', 'sin_th', 'rs_r', 'two_th', 'sin_2th', 'two_r2',
    'two_sigma', 'r4', 'spin4', 'sin2_th', 'spin4_sin2_th',
    'spin2_r2', 'spin2_r2_sin2_th', 'rs_u0', 'r2_plus_spin2',
    'inv_delta_sigma2', 'u1_over_delta_sigma2',
    'rs_u0_u1_over_delta_sigma2', 'spin4_cos2_th',
    'spin2_cos2_th_r2', 'spin_u3', 'spin_sin2_th_u3',
    'inv_two_sigma', 'u2_2', 'r_over_sigma', 'inv_sigma3', 'u0_2',
    'r3', 'rs_r3', 'spin2_cos2_th_spin2_minus_r2_plus_rs_r',
    'radial_poly_a', 'half_rs', 'half_rs_r2',
    'u1_2_over_radial_poly_b', 'r6', 'r5', 'spin6', 'cos4_th',
    'spin6_cos4_th', 'spin4_cos4_th', 'two_spin2_cos2_th',
    'spin4_cos4_th_r2', 'spin2_r4', 'rs_r4', 'one_minus_cos_4th',
    'one_minus_cos_4th_over_16', 'rs_one_minus_cos_4th_over_16',
    'rs2', 'spin2_sin2_th', 'rs_spin4_one_minus_cos_4th_over_16',
    'u3_2_over_sigma3', 'two_u1', 'spin2_sin_2th',
    'rs_r_over_two_sigma_cubed', 'cos_th_sin_th', 'spin4_sin4_th',
    'cot_th', 'two_spin2_r2',
)
assert len(best_cse['replacements']) == len(semantic_names)
assert len(set(semantic_names)) == len(semantic_names)
semantic_symbols = tuple(sp.Symbol(name) for name in semantic_names)
semantic_substitutions = {
    old_symbol: new_symbol
    for (old_symbol, _), new_symbol in zip(
        best_cse['replacements'], semantic_symbols
    )
}
semantic_replacements = tuple(
    (
        semantic_substitutions[old_symbol],
        expression.xreplace(semantic_substitutions),
    )
    for old_symbol, expression in best_cse['replacements']
)
semantic_definitions = dict(semantic_replacements)
semantic_expectations = {
    sp.Symbol('cos_th'): sp.cos(th),
    sp.Symbol('r2'): r**2,
    sp.Symbol('spin2'): spin**2,
    sp.Symbol('cos2_th'): sp.Symbol('cos_th')**2,
    sp.Symbol('sin_th'): sp.sin(th),
    sp.Symbol('rs_r'): rs * r,
    sp.Symbol('r4'): r**4,
    sp.Symbol('spin4'): spin**4,
    sp.Symbol('sin2_th'): sp.Symbol('sin_th')**2,
    sp.Symbol('r3'): r**3,
    sp.Symbol('r5'): r**5,
    sp.Symbol('r6'): r**6,
    sp.Symbol('spin6'): spin**6,
    sp.Symbol('rs2'): rs**2,
    sp.Symbol('cot_th'): 1 / sp.tan(th),
}
assert all(
    sp.simplify(semantic_definitions[symbol] - expected) == 0
    for symbol, expected in semantic_expectations.items()
)
semantic_acceleration = tuple(
    expression.xreplace(semantic_substitutions)
    for expression in best_cse['reduced']
)
summary = [
    {key: value for key, value in item.items() if key not in {'replacements', 'reduced'}}
    for item in strategies
]
{
    'baseline_operations': baseline_operations,
    'selected_cse': {
        'strategies': summary,
        'selected': best_cse['name'],
        'operations': best_cse['operations'],
        'semantic_auxiliaries': len(semantic_replacements),
        'opaque_names': 0,
    },
}

{'baseline_operations': 741,
 'selected_cse': {'strategies': [{'name': 'cse:raw',
    'temporaries': 64,
    'operations': 260,
    'elapsed_s': 0.046982225000647304},
   {'name': 'cse:basic',
    'temporaries': 63,
    'operations': 287,
    'elapsed_s': 0.16240387600009853},
   {'name': 'factor_terms+cse:raw',
    'temporaries': 68,
    'operations': 275,
    'elapsed_s': 0.035232294999332225},
   {'name': 'factor_terms+cse:basic',
    'temporaries': 63,
    'operations': 283,
    'elapsed_s': 0.12151735899988125}],
  'selected': 'cse:raw',
  'operations': 260,
  'semantic_auxiliaries': 64,
  'opaque_names': 0}}

## 4.6. Equivalencia simbólica y numérica

Para auditar el renombrado se expanden los auxiliares semánticos en orden inverso y se compara cada aceleración con la contracción original. Después se evalúan puntos regulares con spin positivo, negativo y nulo. La prueba numérica no sustituye la identidad simbólica; sirve para detectar problemas de generación o evaluación.

In [40]:
def expand_cse(replacements, reduced):
    expanded = list(reduced)
    for symbol, definition in reversed(replacements):
        expanded = [expr.xreplace({symbol: definition}) for expr in expanded]
    return tuple(expanded)


expanded_semantic_cse = expand_cse(semantic_replacements, semantic_acceleration)
symbolic_residuals = tuple(
    sp.simplify(original - optimized)
    for original, optimized in zip(symmetric_acceleration, expanded_semantic_cse)
)
assert symbolic_residuals == (0, 0, 0, 0)

parameters = (rs, spin, r, th, *u)
direct_function = sp.lambdify(parameters, symmetric_acceleration, modules='numpy')
optimized_function = sp.lambdify(parameters, expanded_semantic_cse, modules='numpy')
samples = (
    (2.0, 0.5, 8.0, 1.1, 1.2, -0.03, 0.01, 0.08),
    (4.0, -1.4, 15.0, 0.8, 1.1, 0.02, -0.015, 0.04),
    (1.5, 0.0, 6.0, 1.35, 1.3, -0.01, 0.02, -0.05),
)
max_abs_error = 0.0
for sample in samples:
    direct_value = np.asarray(direct_function(*sample), dtype=float)
    optimized_value = np.asarray(optimized_function(*sample), dtype=float)
    np.testing.assert_allclose(optimized_value, direct_value, rtol=3e-13, atol=3e-14)
    max_abs_error = max(max_abs_error, float(np.max(np.abs(optimized_value - direct_value))))

{'symbolic_residuals': symbolic_residuals, 'max_abs_error': max_abs_error}

{'symbolic_residuals': (0, 0, 0, 0), 'max_abs_error': 0.0}

## 4.7. Emisión de un candidato de kernel C

El bloque generado calcula solamente $du^\mu/d\lambda$. Las subexpresiones comunes se declaran con nombres algebraicos (`r2`, `spin2`, `cos_th`, `sigma`, etc.); nunca con identificadores opacos como `tmp0`. Las primeras cuatro ecuaciones del sistema son triviales: `dx[mu] = u[mu]`. Los parámetros `rs` y `spin` corresponden respectivamente a $R_s=2M$ y al parámetro de spin con dimensión de longitud usado internamente por el legado. Antes de adoptar este texto en producción todavía deben fijarse el vector canónico, la normalización, los dominios y la estrategia de código generado en la arquitectura.

In [41]:
def emit_c_acceleration(replacements, expressions) -> str:
    lines = [
        '/* Generated candidate: verify before adopting in production. */',
        'dx[0] = u0;',
        'dx[1] = u1;',
        'dx[2] = u2;',
        'dx[3] = u3;',
        '',
    ]
    lines.extend(
        f'const double {symbol} = {sp.ccode(expression, standard="C99")};'
        for symbol, expression in replacements
    )
    lines.append('')
    lines.extend(
        f'du[{mu}] = {sp.ccode(expression, standard="C99")};'
        for mu, expression in enumerate(expressions)
    )
    return '\n'.join(lines)


c_kernel = emit_c_acceleration(semantic_replacements, semantic_acceleration)
assert 'tmp' not in c_kernel
assert 'cse_aux' not in c_kernel
assert all(f'const double {name} =' in c_kernel for name in semantic_names)
print(c_kernel)

/* Generated candidate: verify before adopting in production. */
dx[0] = u0;
dx[1] = u1;
dx[2] = u2;
dx[3] = u3;

const double cos_th = cos(th);
const double r2 = pow(r, 2);
const double spin2 = pow(spin, 2);
const double cos2_th = pow(cos_th, 2);
const double spin2_cos2_th = cos2_th*spin2;
const double sigma = r2 + spin2_cos2_th;
const double inv_sigma2 = pow(sigma, -2);
const double sin_th = sin(th);
const double rs_r = r*rs;
const double two_th = 2*th;
const double sin_2th = sin(two_th);
const double two_r2 = 2*r2;
const double two_sigma = spin2*cos(two_th) + spin2 + two_r2;
const double r4 = pow(r, 4);
const double spin4 = pow(spin, 4);
const double sin2_th = pow(sin_th, 2);
const double spin4_sin2_th = sin2_th*spin4;
const double spin2_r2 = r2*spin2;
const double spin2_r2_sin2_th = sin2_th*spin2_r2;
const double rs_u0 = rs*u0;
const double r2_plus_spin2 = r2 + spin2;
const double inv_delta_sigma2 = inv_sigma2/(r2_plus_spin2 - rs_r);
const double u1_over_delta_sigma2 = inv_delta_si

## 4.8. Prueba nativa del código generado

Si hay un compilador `cc`, el fragmento se envuelve en una función temporal, se compila como C99 estricto y se compara mediante `ctypes` con la contracción simbólica directa. No se escribe ningún archivo generado dentro del repositorio.

In [42]:
compiler = shutil.which('cc')
if compiler is None:
    native_validation = {'status': 'SKIP', 'reason': 'cc was not found'}
else:
    wrapper = '''#include <math.h>
void kerr_geodesic_rhs(
    double rs, double spin, double r, double th,
    const double u[4], double dx[4], double du[4]
)
{
    const double u0 = u[0];
    const double u1 = u[1];
    const double u2 = u[2];
    const double u3 = u[3];
'''
    wrapper += '\n'.join(
        f'    {line}' if line else '' for line in c_kernel.splitlines()
    )
    wrapper += '\n}\n'

    with tempfile.TemporaryDirectory(prefix='relatipy-geodesic-') as directory:
        directory = Path(directory)
        source_path = directory / 'kerr_geodesic_rhs.c'
        library_path = directory / 'libkerr_geodesic_rhs.so'
        source_path.write_text(wrapper, encoding='utf-8')
        subprocess.run(
            [
                compiler, '-std=c99', '-shared', '-fPIC',
                '-Wall', '-Wextra', '-Wpedantic', '-Werror',
                str(source_path), '-lm', '-o', str(library_path),
            ],
            check=True,
            capture_output=True,
            text=True,
        )
        library = ctypes.CDLL(str(library_path))
        function = library.kerr_geodesic_rhs
        pointer = ctypes.POINTER(ctypes.c_double)
        function.argtypes = [ctypes.c_double] * 4 + [pointer, pointer, pointer]
        function.restype = None

        native_max_abs_error = 0.0
        for sample in samples:
            sample_rs, sample_spin, sample_r, sample_th, *velocity = sample
            u_buffer = (ctypes.c_double * 4)(*velocity)
            dx_buffer = (ctypes.c_double * 4)()
            du_buffer = (ctypes.c_double * 4)()
            function(
                sample_rs, sample_spin, sample_r, sample_th,
                u_buffer, dx_buffer, du_buffer,
            )
            expected = np.asarray(direct_function(*sample), dtype=float)
            actual = np.ctypeslib.as_array(du_buffer).copy()
            np.testing.assert_allclose(actual, expected, rtol=3e-13, atol=3e-14)
            np.testing.assert_array_equal(np.ctypeslib.as_array(dx_buffer), velocity)
            native_max_abs_error = max(
                native_max_abs_error, float(np.max(np.abs(actual - expected)))
            )

    native_validation = {
        'status': 'PASS',
        'standard': 'C99',
        'samples': len(samples),
        'max_abs_error': native_max_abs_error,
    }

native_validation

{'status': 'PASS',
 'standard': 'C99',
 'samples': 3,
 'max_abs_error': 4.7704895589362195e-18}

## 4.9. Cuadrivelocidad desde velocidades coordenadas

La métrica se extrae de la rama escalar de `Kerr._metric_dimensionless` del mismo snapshot legado. Sean $v^i=dx^i/dt=(v_r,v_\theta,v_\phi)$ y $w^\mu=(1,v_r,v_\theta,v_\phi)$. En las unidades geométricas usadas por estas fórmulas, la rama temporal futura se obtiene de

$$N=g_{\mu\nu}w^\mu w^\nu,\qquad u^0=N^{-1/2},\qquad u^i=v^i u^0.$$

El tensor legado tiene firma $+---$, por lo que la cuadrivelocidad temporal satisface $g_{\mu\nu}u^\mu u^\nu=1$. Se exige $N>0$ y se toma la raíz positiva. Las velocidades físicas deben convertirse al convenio geométrico antes de evaluar este candidato; esta derivación reproducible no cierra la normalización productiva ni el vector canónico que la arquitectura mantiene pendientes.

In [43]:
legacy_metric = extract_legacy_metric(
    LEGACY_KERR, coordinates, spin, rs
)
assert legacy_metric.shape == (4, 4)
assert legacy_metric == legacy_metric.T

v_r, v_th, v_ph = sp.symbols('v_r v_th v_ph', real=True)
coordinate_time_direction = sp.Matrix((1, v_r, v_th, v_ph))
velocity_normalization = sp.factor_terms(
    (coordinate_time_direction.T * legacy_metric * coordinate_time_direction)[0]
)
direct_four_velocity = tuple(
    component / sp.sqrt(velocity_normalization)
    for component in coordinate_time_direction
)

velocity_candidate_inputs = {
    'cse': direct_four_velocity,
    'factor_terms+cse': tuple(
        sp.factor_terms(expression) for expression in direct_four_velocity
    ),
}
velocity_strategies = []
for input_name, expressions in velocity_candidate_inputs.items():
    for optimization_name, optimizations in (('raw', None), ('basic', 'basic')):
        replacements, reduced = sp.cse(
            expressions,
            symbols=sp.numbered_symbols('velocity_cse_aux'),
            optimizations=optimizations,
            order='canonical',
        )
        velocity_strategies.append({
            'name': f'{input_name}:{optimization_name}',
            'replacements': tuple(replacements),
            'reduced': tuple(reduced),
            'auxiliaries': len(replacements),
            'operations': operation_cost(replacements, reduced),
        })

best_velocity_cse = min(
    velocity_strategies,
    key=lambda item: (item['operations'], item['auxiliaries'], item['name']),
)
velocity_semantic_names = (
    'r2', 'spin2', 'sigma', 'rs_r', 'rs_r_over_sigma', 'sin2_th',
    'sin2_th_v_ph', 'rs_r_spin_over_sigma', 'r2_plus_spin2', 'u0',
)
assert len(best_velocity_cse['replacements']) == len(velocity_semantic_names)
velocity_semantic_symbols = tuple(
    sp.Symbol(name) for name in velocity_semantic_names
)
velocity_semantic_substitutions = {
    old_symbol: new_symbol
    for (old_symbol, _), new_symbol in zip(
        best_velocity_cse['replacements'], velocity_semantic_symbols
    )
}
velocity_semantic_replacements = tuple(
    (
        velocity_semantic_substitutions[old_symbol],
        expression.xreplace(velocity_semantic_substitutions),
    )
    for old_symbol, expression in best_velocity_cse['replacements']
)
optimized_four_velocity = tuple(
    expression.xreplace(velocity_semantic_substitutions)
    for expression in best_velocity_cse['reduced']
)

expanded_four_velocity = expand_cse(
    velocity_semantic_replacements, optimized_four_velocity
)
four_velocity_residuals = tuple(
    sp.simplify(direct - optimized)
    for direct, optimized in zip(direct_four_velocity, expanded_four_velocity)
)
normalization_residual = sp.simplify(
    (sp.Matrix(expanded_four_velocity).T
     * legacy_metric
     * sp.Matrix(expanded_four_velocity))[0] - 1
)
velocity_ratio_residuals = tuple(
    sp.simplify(expanded_four_velocity[index] / expanded_four_velocity[0] - velocity)
    for index, velocity in enumerate((v_r, v_th, v_ph), start=1)
)
minkowski_limit_residual = sp.simplify(
    velocity_normalization.subs({rs: 0, spin: 0})
    - (1 - v_r**2 - r**2 * v_th**2 - r**2 * sp.sin(th)**2 * v_ph**2)
)
assert four_velocity_residuals == (0, 0, 0, 0)
assert normalization_residual == 0
assert velocity_ratio_residuals == (0, 0, 0)
assert minkowski_limit_residual == 0

velocity_parameters = (rs, spin, r, th, v_r, v_th, v_ph)
direct_four_velocity_function = sp.lambdify(
    velocity_parameters, direct_four_velocity, modules='numpy'
)
optimized_four_velocity_function = sp.lambdify(
    velocity_parameters, expanded_four_velocity, modules='numpy'
)
normalization_function = sp.lambdify(
    velocity_parameters, velocity_normalization, modules='numpy'
)
velocity_samples = (
    (2.0, 0.5, 8.0, 1.1, -0.01, 0.002, 0.02),
    (4.0, -1.4, 15.0, 0.8, 0.015, -0.001, -0.015),
    (1.5, 0.0, 6.0, 1.35, -0.02, 0.003, 0.01),
)
four_velocity_max_abs_error = 0.0
for sample in velocity_samples:
    assert float(normalization_function(*sample)) > 0.0
    direct_value = np.asarray(direct_four_velocity_function(*sample), dtype=float)
    optimized_value = np.asarray(optimized_four_velocity_function(*sample), dtype=float)
    np.testing.assert_allclose(optimized_value, direct_value, rtol=3e-13, atol=3e-14)
    four_velocity_max_abs_error = max(
        four_velocity_max_abs_error,
        float(np.max(np.abs(optimized_value - direct_value))),
    )

velocity_summary = [
    {key: value for key, value in item.items() if key not in {'replacements', 'reduced'}}
    for item in velocity_strategies
]
{
    'baseline_operations': int(sum(sp.count_ops(expr) for expr in direct_four_velocity)),
    'strategies': velocity_summary,
    'selected': best_velocity_cse['name'],
    'selected_operations': best_velocity_cse['operations'],
    'semantic_auxiliaries': len(velocity_semantic_replacements),
    'normalization_residual': normalization_residual,
    'velocity_ratio_residuals': velocity_ratio_residuals,
    'minkowski_limit_residual': minkowski_limit_residual,
    'max_abs_error': four_velocity_max_abs_error,
}

{'baseline_operations': 332,
 'strategies': [{'name': 'cse:raw', 'auxiliaries': 10, 'operations': 37},
  {'name': 'cse:basic', 'auxiliaries': 10, 'operations': 37},
  {'name': 'factor_terms+cse:raw', 'auxiliaries': 10, 'operations': 37},
  {'name': 'factor_terms+cse:basic', 'auxiliaries': 10, 'operations': 37}],
 'selected': 'cse:basic',
 'selected_operations': 37,
 'semantic_auxiliaries': 10,
 'normalization_residual': 0,
 'velocity_ratio_residuals': (0, 0, 0),
 'minkowski_limit_residual': 0,
 'max_abs_error': 0.0}

## 4.10. C99 semántico para la cuadrivelocidad

El fragmento siguiente recibe las tres velocidades coordenadas ya convertidas al convenio geométrico y escribe $(u^0,u^r,u^\theta,u^\phi)$. Los auxiliares conservan nombres algebraicos y la compilación nativa comprueba los tres casos regulares anteriores. En producción, $N\leq0$, entradas no finitas y dominios singulares deben convertirse en errores tipados antes de evaluar la raíz.

In [44]:
def emit_c_four_velocity(replacements, expressions) -> str:
    lines = [
        '/* Generated candidate: coordinate velocities -> four-velocity. */',
    ]
    lines.extend(
        f'const double {symbol} = {sp.ccode(expression, standard="C99")};'
        for symbol, expression in replacements
    )
    lines.append('')
    lines.extend(
        f'four_velocity[{index}] = {sp.ccode(expression, standard="C99")};'
        for index, expression in enumerate(expressions)
    )
    return '\n'.join(lines)


c_four_velocity_kernel = emit_c_four_velocity(
    velocity_semantic_replacements, optimized_four_velocity
)
assert 'tmp' not in c_four_velocity_kernel
assert 'velocity_cse_aux' not in c_four_velocity_kernel
assert all(
    f'const double {name} =' in c_four_velocity_kernel
    for name in velocity_semantic_names
)

if compiler is None:
    four_velocity_native_validation = {'status': 'SKIP', 'reason': 'cc was not found'}
else:
    wrapper = '''#include <math.h>
void kerr_four_velocity_from_coordinate_velocity(
    double rs, double spin, double r, double th,
    double v_r, double v_th, double v_ph,
    double four_velocity[4]
)
{
'''
    wrapper += '\n'.join(
        f'    {line}' if line else ''
        for line in c_four_velocity_kernel.splitlines()
    )
    wrapper += '\n}\n'

    with tempfile.TemporaryDirectory(prefix='relatipy-four-velocity-') as directory:
        directory = Path(directory)
        source_path = directory / 'kerr_four_velocity.c'
        library_path = directory / 'libkerr_four_velocity.so'
        source_path.write_text(wrapper, encoding='utf-8')
        subprocess.run(
            [
                compiler, '-std=c99', '-shared', '-fPIC',
                '-Wall', '-Wextra', '-Wpedantic', '-Werror',
                str(source_path), '-lm', '-o', str(library_path),
            ],
            check=True,
            capture_output=True,
            text=True,
        )
        library = ctypes.CDLL(str(library_path))
        function = library.kerr_four_velocity_from_coordinate_velocity
        pointer = ctypes.POINTER(ctypes.c_double)
        function.argtypes = [ctypes.c_double] * 7 + [pointer]
        function.restype = None

        four_velocity_native_max_abs_error = 0.0
        for sample in velocity_samples:
            output = (ctypes.c_double * 4)()
            function(*sample, output)
            expected = np.asarray(direct_four_velocity_function(*sample), dtype=float)
            actual = np.ctypeslib.as_array(output).copy()
            np.testing.assert_allclose(actual, expected, rtol=3e-13, atol=3e-14)
            four_velocity_native_max_abs_error = max(
                four_velocity_native_max_abs_error,
                float(np.max(np.abs(actual - expected))),
            )

    four_velocity_native_validation = {
        'status': 'PASS',
        'standard': 'C99',
        'samples': len(velocity_samples),
        'max_abs_error': four_velocity_native_max_abs_error,
    }

print(c_four_velocity_kernel)
four_velocity_native_validation

/* Generated candidate: coordinate velocities -> four-velocity. */
const double r2 = pow(r, 2);
const double spin2 = pow(spin, 2);
const double sigma = r2 + spin2*pow(cos(th), 2);
const double rs_r = r*rs;
const double rs_r_over_sigma = rs_r/sigma;
const double sin2_th = pow(sin(th), 2);
const double sin2_th_v_ph = sin2_th*v_ph;
const double rs_r_spin_over_sigma = rs_r_over_sigma*spin;
const double r2_plus_spin2 = r2 + spin2;
const double u0 = pow(-rs_r_over_sigma + rs_r_spin_over_sigma*sin2_th_v_ph - sigma*pow(v_r, 2)/(r2_plus_spin2 - rs_r) - sigma*pow(v_th, 2) + sin2_th_v_ph*(rs_r_spin_over_sigma - v_ph*(r2_plus_spin2 + rs_r_over_sigma*sin2_th*spin2)) + 1, -1.0/2.0);

four_velocity[0] = u0;
four_velocity[1] = u0*v_r;
four_velocity[2] = u0*v_th;
four_velocity[3] = u0*v_ph;


{'status': 'PASS',
 'standard': 'C99',
 'samples': 3,
 'max_abs_error': 4.336808689942018e-19}

## 4.11. Criterios antes de integrar los kernels

El candidato solo debería trasladarse a `relatipy/native/` después de:

1. congelar la decisión arquitectónica sobre el estado canónico y la normalización;
2. generar el archivo C mediante un procedimiento reproducible, sin copiar manualmente expresiones;
3. comparar el RHS optimizado con la contracción directa en una malla que cubra Schwarzschild, Kerr, ambos signos del spin y cercanía controlada a superficies de coordenadas;
4. validar la cuadrivelocidad en límites analíticos, frente al legado y ante dominios no temporales o no finitos;
5. comprobar geodésicas completas frente a KerrGeoPy y PyGRO dentro de sus dominios aplicables;
6. medir rendimiento con compilador, flags, hardware y tolerancias registrados; y
7. conservar este evaluador simbólico como referencia auditable.

Los registros científicos locales INF-006, INF-007 e INF-013 siguen con `manual_review: pending`; sus comparaciones no deben presentarse como verificación humana concluida.